# 🔬 Cancer Detection using CNN
### Transfer Learning with EfficientNetB0
**Dataset:** Histopathologic Cancer Detection (Kaggle)
**Task:** Binary Classification — Cancerous vs Non-Cancerous Tissue

## 📦 Step 1 — Install Dependencies

In [ ]:
!pip install kaggle -q
import os, numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve
from sklearn.utils.class_weight import compute_class_weight
import tensorflow as tf
from tensorflow.keras import layers, models, optimizers, callbacks
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import warnings; warnings.filterwarnings('ignore')
print("TensorFlow:", tf.__version__)

## 🔑 Step 2 — Upload kaggle.json API Key

In [ ]:
from google.colab import files
files.upload()   # upload your kaggle.json here

os.makedirs('/root/.kaggle', exist_ok=True)
os.rename('kaggle.json', '/root/.kaggle/kaggle.json')
os.chmod('/root/.kaggle/kaggle.json', 0o600)
print("✅ Kaggle API key configured!")

## 📥 Step 3 — Download Dataset
> ⚠️ First accept competition rules at: https://www.kaggle.com/competitions/histopathologic-cancer-detection/rules

In [ ]:
!kaggle competitions download -c histopathologic-cancer-detection -p /content/
!unzip -q /content/histopathologic-cancer-detection.zip -d /content/data/
!ls /content/data/

## ⚙️ Step 4 — Configuration

In [ ]:
DATA_DIR   = '/content/data'
TRAIN_DIR  = os.path.join(DATA_DIR, 'train')
LABELS_CSV = os.path.join(DATA_DIR, 'train_labels.csv')
IMG_SIZE   = 96; BATCH_SIZE = 64; EPOCHS = 20; LEARNING_RATE = 1e-4
USE_SUBSET = True; SUBSET_FRAC = 0.20; SEED = 42
tf.random.set_seed(SEED); np.random.seed(SEED)

## 📁 Step 5 — Load & Explore Labels

In [ ]:
df = pd.read_csv(LABELS_CSV)
df['filename'] = df['id'] + '.tif'
df['label'] = df['label'].astype(str)
print(f"Total samples: {len(df)}")
print(df['label'].value_counts())

plt.figure(figsize=(6,4))
df['label'].value_counts().plot(kind='bar', color=['steelblue','crimson'], rot=0)
plt.title('Class Distribution'); plt.xlabel('Label'); plt.ylabel('Count')
plt.xticks([0,1], ['Non-Cancerous (0)','Cancerous (1)'])
plt.tight_layout(); plt.show()

## 🔄 Step 6 — Prepare DataFrames (with optional subset)

In [ ]:
if USE_SUBSET:
    df, _ = train_test_split(df, test_size=1-SUBSET_FRAC, stratify=df['label'], random_state=SEED)
    print(f"Using subset: {len(df)} samples ({SUBSET_FRAC*100:.0f}%)")

train_df, test_df = train_test_split(df, test_size=0.15, stratify=df['label'], random_state=SEED)
train_df, val_df  = train_test_split(train_df, test_size=0.1, stratify=train_df['label'], random_state=SEED)
print(f"Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")

## 📁 Step 7 — Data Generators + Augmentation

In [ ]:
train_datagen = ImageDataGenerator(rescale=1/255, rotation_range=180,
    horizontal_flip=True, vertical_flip=True,
    width_shift_range=0.1, height_shift_range=0.1,
    zoom_range=0.1, fill_mode='reflect')
vt_datagen = ImageDataGenerator(rescale=1/255)

train_gen = train_datagen.flow_from_dataframe(train_df, directory=TRAIN_DIR,
    x_col='filename', y_col='label', target_size=(IMG_SIZE,IMG_SIZE),
    batch_size=BATCH_SIZE, class_mode='binary', seed=SEED)
val_gen   = vt_datagen.flow_from_dataframe(val_df, directory=TRAIN_DIR,
    x_col='filename', y_col='label', target_size=(IMG_SIZE,IMG_SIZE),
    batch_size=BATCH_SIZE, class_mode='binary', shuffle=False)
test_gen  = vt_datagen.flow_from_dataframe(test_df, directory=TRAIN_DIR,
    x_col='filename', y_col='label', target_size=(IMG_SIZE,IMG_SIZE),
    batch_size=BATCH_SIZE, class_mode='binary', shuffle=False)

## 🔍 Step 8 — Visualize Sample Patches

In [ ]:
X_b, y_b = next(train_gen)
fig, axes = plt.subplots(2, 8, figsize=(18, 6))
for i, ax in enumerate(axes.flat):
    ax.imshow(X_b[i])
    lbl = 'Cancer' if y_b[i]==1 else 'Normal'
    ax.set_title(lbl, color='red' if y_b[i]==1 else 'green', fontsize=8); ax.axis('off')
plt.suptitle('Sample Histopathology Patches (96×96)', fontsize=13)
plt.tight_layout(); plt.show()

## 🏗️ Step 9 — Build Model (EfficientNetB0 + Custom Head)

In [ ]:
base = EfficientNetB0(weights='imagenet', include_top=False, input_shape=(IMG_SIZE,IMG_SIZE,3))
for layer in base.layers[:-30]: layer.trainable = False

inp = tf.keras.Input(shape=(IMG_SIZE,IMG_SIZE,3))
x   = base(inp, training=False)
x   = layers.GlobalAveragePooling2D()(x)
x   = layers.Dense(256, activation='relu')(x)
x   = layers.BatchNormalization()(x)
x   = layers.Dropout(0.5)(x)
x   = layers.Dense(64, activation='relu')(x)
x   = layers.Dropout(0.3)(x)
out = layers.Dense(1, activation='sigmoid')(x)

model = tf.keras.Model(inp, out)
model.compile(optimizer=optimizers.Adam(LEARNING_RATE),
              loss='binary_crossentropy',
              metrics=['accuracy', tf.keras.metrics.AUC(name='auc')])
model.summary()

## ⚖️ Step 10 — Class Weights (handle imbalance)

In [ ]:
lbl_arr = train_df['label'].astype(int).values
cw = compute_class_weight('balanced', classes=np.unique(lbl_arr), y=lbl_arr)
class_weights = {0: cw[0], 1: cw[1]}
print(f"Class weights: {class_weights}")

## 🚀 Step 11 — Train

In [ ]:
cb_list = [
    callbacks.EarlyStopping(monitor='val_auc', patience=5,
                             restore_best_weights=True, mode='max', verbose=1),
    callbacks.ReduceLROnPlateau(monitor='val_auc', factor=0.5,
                                 patience=3, mode='max', verbose=1),
    callbacks.ModelCheckpoint('/content/cancer_best.h5', monitor='val_auc',
                               save_best_only=True, mode='max', verbose=1)
]
history = model.fit(train_gen, validation_data=val_gen, epochs=EPOCHS,
                    class_weight=class_weights, callbacks=cb_list)

## 📈 Step 12 — Training Curves

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, metric in zip(axes, ['accuracy', 'loss', 'auc']):
    ax.plot(history.history[metric], label='Train')
    ax.plot(history.history[f'val_{metric}'], label='Validation')
    ax.set_title(metric.capitalize()); ax.set_xlabel('Epoch'); ax.legend()
plt.suptitle('Cancer Detection — Training History', fontsize=14)
plt.tight_layout(); plt.show()

## 🧪 Step 13 — Evaluate

In [ ]:
res = model.evaluate(test_gen, verbose=0)
print(f"\n✅ Test Accuracy: {res[1]*100:.2f}%  |  AUC: {res[2]:.4f}")

test_gen.reset()
y_prob = model.predict(test_gen).flatten()
y_pred = (y_prob >= 0.5).astype(int)
y_true = test_gen.labels.astype(int)
print("\n📊 Classification Report:")
print(classification_report(y_true, y_pred, target_names=['Non-Cancerous','Cancerous']))

In [ ]:
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(7,6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Reds',
            xticklabels=['Non-Cancerous','Cancerous'],
            yticklabels=['Non-Cancerous','Cancerous'])
plt.title('Confusion Matrix — Cancer Detection', fontsize=14)
plt.ylabel('True Label'); plt.xlabel('Predicted Label')
plt.tight_layout(); plt.show()

In [ ]:
fpr, tpr, _ = roc_curve(y_true, y_prob)
auc = roc_auc_score(y_true, y_prob)
plt.figure(figsize=(7,6))
plt.plot(fpr, tpr, color='crimson', lw=2, label=f'ROC Curve (AUC = {auc:.4f})')
plt.plot([0,1],[0,1],'k--',lw=1)
plt.xlabel('False Positive Rate'); plt.ylabel('True Positive Rate')
plt.title('ROC Curve — Cancer Detection')
plt.legend(loc='lower right'); plt.tight_layout(); plt.show()

In [ ]:
model.save('/content/cancer_detection_final.h5')
print("✅ Model saved to /content/cancer_detection_final.h5")